In [ ]:
from datasets import load_dataset

def get_gsm8k(split="train"):
    dataset = load_dataset("gsm8k", "main", split=split)
    return dataset

In [ ]:
ds = get_gsm8k("train")

print(ds[0])

import re

def extract_answer(answer_str):
    match = re.search(r"####\s*(-?\d+)", answer_str)
    if match:
        return int(match.group(1))
    return None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [ ]:
ds[150]['answer']

'Let p be the number of packages Angela delivers and m be the number of meals. We know that p + m = 27 and p = 8m.\nSubstituting the second equation into the first equation, we get 8m + m = 27\nCombining like terms, we get 9m = 27\nDividing both sides by 9, we get m = 3\n#### 3'

In [ ]:
ds[150]['question']

'Angela is a bike messenger in New York. She needs to deliver 8 times as many packages as meals. If she needs to deliver 27 meals and packages combined, how many meals does she deliver?'

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # closest available small instruct

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [ ]:
embedding_matrix = model.get_input_embeddings().weight
print(embedding_matrix.shape)


torch.Size([151936, 896])


In [ ]:
prompt = "What is 2+2?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True
    )

In [ ]:
# Install if needed:
# pip install datasets

from datasets import load_dataset

# Load MATH-500
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")

print(dataset)
print(dataset.column_names)

# Show the first few examples
for i in range(5):
    example = dataset[i]

    print("=" * 80)
    print(f"Example {i}")
    print("-" * 80)

    print("QUESTION:")
    print(example["problem"])

    print("\nANSWER / SOLUTION:")
    print(example["solution"])

    # Some versions also include a final answer field
    if "answer" in example:
        print("\nFINAL ANSWER:")
        print(example["answer"])

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
    num_rows: 500
})
['problem', 'solution', 'answer', 'subject', 'level', 'unique_id']
Example 0
--------------------------------------------------------------------------------
QUESTION:
Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$

ANSWER / SOLUTION:
We have that $r = \sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\frac{\pi}{2}$ with the positive $x$-axis.

[asy]
unitsize(0.8 cm);

draw((-0.5,0)--(3.5,0));
draw((0,-0.5)--(0,3.5));
draw(arc((0,0),3,0,90),red,Arrow(6));

dot((0,3), red);
label("$(0,3)$", (0,3), W);
dot((3,0), red);
[/asy]

Therefore, the polar coordinates are $\boxed{\left( 3, \frac{\pi}{2} \right)}.$

FINAL ANSWER:
\left( 3, \frac{\pi}{2} \right)
Example 1
----------------------------------------

In [ ]:
outputs['hidden_states'][-1].shape

torch.Size([1, 7, 896])

In [ ]:
outputs['logits'].shape

torch.Size([1, 7, 151936])

In [ ]:
temperature = 0.7

In [ ]:
h_final = outputs.hidden_states[-1]   # shape (B, T, D)
logits = model.lm_head(h_final)[:,-1,:] #Get the final logits layer

In [ ]:
import torch.nn as nn
import math
import torch.nn.functional as F
class HRPOGate(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_a = nn.Linear(hidden_dim, hidden_dim)
        self.W_x = nn.Linear(hidden_dim, hidden_dim)
        self.Lambda = nn.Parameter(torch.rand(hidden_dim))  # [0,1]
        self.c = 8.0
        self.init_linear(self.W_a)
        self.init_linear(self.W_x)

    def init_linear(self,layer):
      hidden_dim = layer.weight.size(1)  # input dim

      bound = 1 / math.sqrt(hidden_dim)

      nn.init.uniform_(layer.weight, -bound, bound)

      if layer.bias is not None:
          nn.init.uniform_(layer.bias, -bound, bound)

    def forward(self, e_token, h_proj,think):
        # e_token: (B, D)
        # h_proj: (B, D)

        r = torch.sigmoid(self.W_a(e_token))
        i = torch.sigmoid(self.W_x(e_token))

        a = torch.exp(-self.c * F.softplus(self.Lambda) * r)
        mix = torch.sqrt(torch.clamp(1 - a**2, min=1e-8))
        if think == True:
          e_next = a * e_token + mix * (i * h_proj)
        else:
          e_next = e_token
        return e_next

In [ ]:
SYSTEM_PROMPT = (
    """A conversation between User and Assistant. The user asks a question,
and the assistant solves it. The assistant first thinks about the
reasoning process in the mind and then provides the user with the
answer. The final answer is provided after the #### tag, i.e.,
{reasoning process} #### {answer}."""
)

def build_prompt(question: str) -> str:
    return (
        "<|system|>\n"
        f"{SYSTEM_PROMPT}\n"
        "<|user|>\n"
        f"{question}\n"
        "<|assistant|>\n"
    )

In [ ]:
def build_optimizer(policy):
    lora_params = []
    gate_linear_params = []
    lambda_params = []

    for name, p in policy.named_parameters():
        if not p.requires_grad:
            continue

        if "lora_" in name:
            lora_params.append(p)
        elif name.endswith("gate.Lambda"):
            lambda_params.append(p)
        elif name.startswith("gate."):
            gate_linear_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": lora_params, "lr": 5e-6, "weight_decay": 0.1},
            {"params": gate_linear_params, "lr": 1e-4, "weight_decay": 0.1},
            {"params": lambda_params, "lr": 1e-3, "weight_decay": 0.1},
        ],
        betas=(0.9, 0.99),
    )
    return optimizer

In [ ]:
class HybridReasoningPolicy(nn.Module):
    def __init__(self, qwen_model, tokenizer, gate):
        super().__init__()
        self.model = qwen_model
        self.tokenizer = tokenizer
        self.gate = gate

    @property
    def embed_tokens(self):
        return self.model.model.embed_tokens

    @property
    def lm_head(self):
        return self.model.lm_head

    def project_hidden_to_embedding(self, last_hidden, tau=1.0):
        logits = self.lm_head(last_hidden)                  # (B, V)
        probs = torch.softmax(logits / tau, dim=-1)        # (B, V)
        probs = probs / probs.norm(dim=-1, keepdim=True).clamp_min(1e-8)

        W_e = self.embed_tokens.weight.to(probs.dtype)     # (V, D)
        projected_hidden = probs @ W_e                     # (B, D)
        return logits, probs, projected_hidden

    def forward_base(self, input_ids=None, attention_mask=None, inputs_embeds=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            inputs_embeds=inputs_embeds,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )
h = HRPOGate(896).to('cuda')
policy = HybridReasoningPolicy(model, tokenizer, h)

In [ ]:
import re
from datasets import load_dataset


def get_gsm8k(split="train"):
    return load_dataset("openai/gsm8k", "main", split=split)


def extract_answer(answer_str):
    match = re.search(r"####\s*(-?\d+)", answer_str)
    if match:
        return int(match.group(1))
    return None


def extract_model_answer(text):
    """
    Extract final answer from model text.
    We look for #### number first, then fallback to last integer.
    """
    match = re.search(r"####\s*(-?\d+)", text)
    if match:
        return int(match.group(1))

    nums = re.findall(r"-?\d+", text)
    if len(nums) > 0:
        return int(nums[-1])

    return None


def reward_fn(model_text, gt_answer):
    pred = extract_model_answer(model_text)

    if pred is None:
        return 0.0

    if pred == gt_answer:
        return 1.0

    return 0.0

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any


@dataclass
class Rollout:
    prompt: str
    target_answer: str
    generated_text: str
    generated_token_ids: List[int]
    reward: float
    advantage: float = 0.0

In [ ]:
def top_p_filtering(logits, top_p=0.9, filter_value=-float("inf")):
    """
    logits: (B, V)
    """
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

    sorted_indices_to_remove = cumulative_probs > top_p

    # Keep at least the first token
    sorted_indices_to_remove[..., 0] = False

    indices_to_remove = sorted_indices_to_remove.scatter(
        dim=-1,
        index=sorted_indices,
        src=sorted_indices_to_remove,
    )

    logits = logits.masked_fill(indices_to_remove, filter_value)
    return logits

@torch.no_grad()
def hybrid_generate_one(
    policy: HybridReasoningPolicy,
    prompt: str,
    max_new_tokens: int = 256,
    temperature: float = 1.0,
    top_p: float = 1.0,
    tau_projection: float = 1.0,
):
    device = next(policy.parameters()).device
    tokenizer = policy.tokenizer

    encoded = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = encoded["input_ids"]                    # (1, T)
    attention_mask = encoded["attention_mask"]          # (1, T)

    generated_ids = []

    # Start with normal prompt embeddings
    inputs_embeds = policy.embed_tokens(input_ids)      # (1, T, D)

    think_mode = False

    for step in range(max_new_tokens):
        outputs = policy.forward_base(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
        )

        last_hidden = outputs.hidden_states[-1][:, -1, :]      # (1, D)

        logits, probs, h_proj = policy.project_hidden_to_embedding(
            last_hidden,
            tau=tau_projection,
        )

        next_logits = logits[:, -1, :] if logits.dim() == 3 else logits
        next_logits = next_logits / temperature

        # optional top-p
        if top_p < 1.0:
            next_logits = top_p_filtering(next_logits, top_p=top_p)

        dist = torch.distributions.Categorical(logits=next_logits)
        next_token = dist.sample()                            # (1,)

        token_id = next_token.item()
        generated_ids.append(token_id)

        token_text = tokenizer.decode([token_id])

        # Track whether we are inside <think>

        think_mode = True


        # Stop at EOS
        if token_id == tokenizer.eos_token_id:
            break

        e_token = policy.embed_tokens(next_token.view(1, 1))[:, 0, :]  # (1, D)

        e_next = policy.gate(
            e_token=e_token,
            h_proj=h_proj,
            think=think_mode,
        )                                                             # (1, D)

        # Append custom embedding
        inputs_embeds = torch.cat(
            [inputs_embeds, e_next.unsqueeze(1)],
            dim=1,
        )

        new_mask = torch.ones((1, 1), device=device, dtype=attention_mask.dtype)
        attention_mask = torch.cat([attention_mask, new_mask], dim=1)

    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return generated_text, generated_ids

In [ ]:
ques = build_prompt(ds['question'][0])

In [ ]:
ques

'<|system|>\nA conversation between User and Assistant. The user asks a question,\nand the assistant solves it. The assistant first thinks about the\nreasoning process in the mind and then provides the user with the\nanswer. The final answer is provided after the #### tag, i.e.,\n{reasoning process} #### {answer}.\n<|user|>\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\n<|assistant|>\n'

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
policy = policy.to(device)
device = policy.embed_tokens.weight.device
dtype = policy.embed_tokens.weight.dtype

policy.gate = policy.gate.to(
    device=policy.embed_tokens.weight.device,
    dtype=policy.embed_tokens.weight.dtype,
)

print("embed:", policy.embed_tokens.weight.device, policy.embed_tokens.weight.dtype)
print("gate :", next(policy.gate.parameters()).device, next(policy.gate.parameters()).dtype)
text,ids = hybrid_generate_one(policy,ques,max_new_tokens = 128,temperature = 0.7)

embed: cuda:0 torch.bfloat16
gate : cuda:0 torch.bfloat16


In [ ]:
def compute_rollout_logprob_and_kl(
    policy: HybridReasoningPolicy,
    ref_policy: HybridReasoningPolicy,
    rollout: Rollout,
    beta_kl: float = 0.02,
    tau_projection: float = 1.0,
):
    """
    Replays one rollout token-by-token and computes:

        policy_logprob = sum log pi_theta(a_t | state_t)
        ref_logprob    = sum log pi_ref(a_t | state_t)
        approx_kl      = policy_logprob - ref_logprob

    For first version, this uses the same hybrid state construction for policy.
    Reference model should be frozen.
    """

    device = next(policy.parameters()).device
    tokenizer = policy.tokenizer

    encoded = tokenizer(rollout.prompt, return_tensors="pt").to(device)

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    inputs_embeds = policy.embed_tokens(input_ids)

    generated_ids = rollout.generated_token_ids

    total_logprob = torch.tensor(0.0, device=device)
    total_ref_logprob = torch.tensor(0.0, device=device)

    think_mode = "<think>" in rollout.prompt and "</think>" not in rollout.prompt

    for token_id in generated_ids:
        token_tensor = torch.tensor([token_id], device=device, dtype=torch.long)

        # -------------------------
        # Current trainable policy
        # -------------------------
        outputs = policy.forward_base(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
        )

        last_hidden = outputs.hidden_states[-1][:, -1, :]  # (1, D)

        logits, probs, h_proj = policy.project_hidden_to_embedding(
            last_hidden,
            tau=tau_projection,
        )

        next_logits = logits[:, -1, :] if logits.dim() == 3 else logits  # (1, V)

        log_probs = F.log_softmax(next_logits, dim=-1)
        token_logprob = log_probs[0, token_id]

        total_logprob = total_logprob + token_logprob

        # -------------------------
        # Frozen reference policy
        # -------------------------
        with torch.no_grad():
            ref_outputs = ref_policy.forward_base(
                inputs_embeds=inputs_embeds.detach(),
                attention_mask=attention_mask,
            )

            ref_last_hidden = ref_outputs.hidden_states[-1][:, -1, :]

            ref_logits, _, _ = ref_policy.project_hidden_to_embedding(
                ref_last_hidden,
                tau=tau_projection,
            )

            ref_next_logits = ref_logits[:, -1, :] if ref_logits.dim() == 3 else ref_logits

            ref_log_probs = F.log_softmax(ref_next_logits, dim=-1)
            ref_token_logprob = ref_log_probs[0, token_id]

            total_ref_logprob = total_ref_logprob + ref_token_logprob

        token_text = tokenizer.decode([token_id])

        if "<think>" in token_text:
            think_mode = True
        if "</think>" in token_text:
            think_mode = False

        # Use the sampled token to build next state
        e_token = policy.embed_tokens(token_tensor.view(1, 1))[:, 0, :]  # (1, D)

        e_next = policy.gate(
            e_token=e_token,
            h_proj=h_proj,
            think=think_mode,
        )

        inputs_embeds = torch.cat(
            [inputs_embeds, e_next.unsqueeze(1)],
            dim=1,
        )

        new_mask = torch.ones((1, 1), device=device, dtype=attention_mask.dtype)
        attention_mask = torch.cat([attention_mask, new_mask], dim=1)

    approx_kl = total_logprob - total_ref_logprob

    return total_logprob, approx_kl

In [ ]:
extract_model_answer(text)

72

In [ ]:
text

'In find the total number of clips Natal in April and May, we need the the total of clips sold in both month,Aprilatalia sold clips to 48 friends her friends in April, and then she sold half as many clips in May, Therefore in she number sold in May is 48/2 = 24 clips. Therefore, total total number of clips sold in April and May is 48+ 24 = 72 clips. Therefore, the answer is {72 clips.####reason} #### 72. {reasoning process}#### Natalia sold clips in 48'

In [ ]:
import copy
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

policy = policy.to(device)

# Make gate match model dtype
policy.gate.to(
    device=policy.embed_tokens.weight.device,
    dtype=policy.embed_tokens.weight.dtype,
)

# Frozen reference copy
ref_policy = copy.deepcopy(policy).to(device)
ref_policy.eval()

for p in ref_policy.parameters():
    p.requires_grad = False

print("policy embed:", policy.embed_tokens.weight.device, policy.embed_tokens.weight.dtype)
print("policy gate :", next(policy.gate.parameters()).device, next(policy.gate.parameters()).dtype)
print("ref embed   :", ref_policy.embed_tokens.weight.device, ref_policy.embed_tokens.weight.dtype)

policy embed: cuda:0 torch.bfloat16
policy gate : cuda:0 torch.bfloat16
ref embed   : cuda:0 torch.bfloat16


In [ ]:
def compute_policy_loss(
    policy,
    ref_policy,
    rollout,
    reward,
    baseline=0.0,
    beta_kl=0.02,
    tau_projection=1.0,
):
    total_logprob, approx_kl = compute_rollout_logprob_and_kl(
        policy=policy,
        ref_policy=ref_policy,
        rollout=rollout,
        tau_projection=tau_projection,
    )

    advantage = reward - baseline

    policy_loss = -advantage * total_logprob
    kl_loss = beta_kl * approx_kl

    loss = policy_loss + kl_loss

    metrics = {
        "loss": loss.detach(),
        "policy_loss": policy_loss.detach(),
        "kl_loss": kl_loss.detach(),
        "total_logprob": total_logprob.detach(),
        "approx_kl": approx_kl.detach(),
        "advantage": torch.as_tensor(advantage).detach(),
    }

    return loss, metrics

In [ ]:
rollout = Rollout(
    prompt=prompt,
    target_answer=answer,
    generated_text=text,
    generated_token_ids=ids,
    reward=compute_reward(text, answer),
    advantage=1.0,  # temporary for testing
)

NameError: name 'answer' is not defined

In [ ]:
def compute_reward(pred_answer, gold_answer, num_generated_tokens):
    correct = normalize_answer(pred_answer) == normalize_answer(gold_answer)

    accuracy_reward = 1.0 if correct else 0.0

    # Small penalty for being too verbose
    length_penalty = 0.001 * num_generated_tokens

    reward = accuracy_reward - length_penalty

    return reward

In [ ]:
import torch
import torch.nn.functional as F


def compute_rollout_logprob_and_kl(
    policy,
    ref_policy,
    rollout,
    tau_projection: float = 1.0,
):
    """
    Replay one rollout token-by-token.

    Computes:
        total_logprob      = sum_t log pi_theta(a_t | history_t)
        total_ref_logprob  = sum_t log pi_ref(a_t | history_t)
        approx_kl          = total_logprob - total_ref_logprob

    The policy uses the hybrid gated embedding update.
    The reference model uses normal token embeddings.
    """

    device = next(policy.parameters()).device
    tokenizer = policy.tokenizer

    encoded = tokenizer(
        rollout.prompt,
        return_tensors="pt",
    ).to(device)

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # Policy state
    policy_inputs_embeds = policy.embed_tokens(input_ids)

    # Reference state
    ref_inputs_embeds = ref_policy.embed_tokens(input_ids)

    generated_ids = rollout.generated_token_ids

    total_logprob = torch.tensor(0.0, device=device)
    total_ref_logprob = torch.tensor(0.0, device=device)

    think_mode = False
    generated_text_so_far = ""

    for token_id in generated_ids:
        token_tensor = torch.tensor([[token_id]], device=device, dtype=torch.long)

        # ============================================================
        # 1. Current trainable policy logprob
        # ============================================================
        policy_outputs = policy.forward_base(
            inputs_embeds=policy_inputs_embeds,
            attention_mask=attention_mask,
        )

        policy_last_hidden = policy_outputs.hidden_states[-1][:, -1, :]  # (1, D)

        policy_logits, _, h_proj = policy.project_hidden_to_embedding(
            policy_last_hidden,
            tau=tau_projection,
        )

        if policy_logits.dim() == 3:
            policy_next_logits = policy_logits[:, -1, :]
        else:
            policy_next_logits = policy_logits

        policy_log_probs = F.log_softmax(policy_next_logits, dim=-1)
        token_logprob = policy_log_probs[0, token_id]

        total_logprob = total_logprob + token_logprob

        # ============================================================
        # 2. Frozen reference logprob
        # ============================================================
        with torch.no_grad():
            ref_outputs = ref_policy.forward_base(
                inputs_embeds=ref_inputs_embeds,
                attention_mask=attention_mask,
            )

            ref_last_hidden = ref_outputs.hidden_states[-1][:, -1, :]

            ref_logits, _, _ = ref_policy.project_hidden_to_embedding(
                ref_last_hidden,
                tau=tau_projection,
            )

            if ref_logits.dim() == 3:
                ref_next_logits = ref_logits[:, -1, :]
            else:
                ref_next_logits = ref_logits

            ref_log_probs = F.log_softmax(ref_next_logits, dim=-1)
            ref_token_logprob = ref_log_probs[0, token_id]

            total_ref_logprob = total_ref_logprob + ref_token_logprob

        # ============================================================
        # 3. Update think mode
        # ============================================================
        generated_text_so_far += tokenizer.decode([token_id])

        # Simple robust-ish check
        last_open = generated_text_so_far.rfind("<think>")
        last_close = generated_text_so_far.rfind("</think>")

        think_mode = last_open > last_close

        # ============================================================
        # 4. Append generated token to policy state
        # ============================================================
        policy_e_token = policy.embed_tokens(token_tensor)[:, 0, :]  # (1, D)

        policy_e_next = policy.gate(
            e_token=policy_e_token,
            h_proj=h_proj,
            think=think_mode,
        )

        policy_inputs_embeds = torch.cat(
            [policy_inputs_embeds, policy_e_next.unsqueeze(1)],
            dim=1,
        )

        # ============================================================
        # 5. Append generated token to reference state
        # ============================================================
        ref_e_next = ref_policy.embed_tokens(token_tensor)  # (1, 1, D)

        ref_inputs_embeds = torch.cat(
            [ref_inputs_embeds, ref_e_next],
            dim=1,
        )

        # ============================================================
        # 6. Update attention mask
        # ============================================================
        new_mask = torch.ones(
            (1, 1),
            device=device,
            dtype=attention_mask.dtype,
        )

        attention_mask = torch.cat([attention_mask, new_mask], dim=1)

    approx_kl = total_logprob - total_ref_logprob

    return total_logprob, approx_kl

In [ ]:
from dataclasses import dataclass
import torch


@dataclass
class Rollout:
    prompt: str
    generated_token_ids: list[int]
    text: str


@torch.no_grad()
def generate_rollout(
    policy,
    prompt,
    max_new_tokens=256,
    temperature=1.0,
    top_p=0.95,
):
    policy.eval()

    device = next(policy.parameters()).device
    tokenizer = policy.tokenizer

    encoded = tokenizer(prompt, return_tensors="pt").to(device)

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    inputs_embeds = policy.embed_tokens(input_ids)

    generated_token_ids = []
    generated_text_so_far = ""

    think_mode = False

    for _ in range(max_new_tokens):
        outputs = policy.forward_base(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
        )

        last_hidden = outputs.hidden_states[-1][:, -1, :]

        logits, probs, h_proj = policy.project_hidden_to_embedding(
            last_hidden,
            tau=1.0,
        )

        if logits.dim() == 3:
            next_logits = logits[:, -1, :]
        else:
            next_logits = logits

        next_logits = next_logits / temperature

        probs = torch.softmax(next_logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)  # (1, 1)
        token_id = next_token.item()

        generated_token_ids.append(token_id)

        token_text = tokenizer.decode([token_id])
        generated_text_so_far += token_text

        if token_id == tokenizer.eos_token_id:
            break

        last_open = generated_text_so_far.rfind("<think>")
        last_close = generated_text_so_far.rfind("</think>")
        think_mode = last_open > last_close

        e_token = policy.embed_tokens(next_token)[:, 0, :]

        e_next = policy.gate(
            e_token=e_token,
            h_proj=h_proj,
            think=think_mode,
        )

        inputs_embeds = torch.cat(
            [inputs_embeds, e_next.unsqueeze(1)],
            dim=1,
        )

        new_mask = torch.ones(
            (1, 1),
            device=device,
            dtype=attention_mask.dtype,
        )

        attention_mask = torch.cat([attention_mask, new_mask], dim=1)

    return Rollout(
        prompt=prompt,
        generated_token_ids=generated_token_ids,
        text=generated_text_so_far,
    )

In [ ]:
import re
from datasets import Dataset


SYSTEM_PROMPT = (
    """A conversation between User and Assistant. The user asks a question,
and the assistant solves it. The assistant first thinks about the
reasoning process in the mind and then provides the user with the
answer. The final answer is provided after the #### tag, i.e.,
{reasoning process} #### {answer}."""
)


def build_prompt(question: str) -> str:
    return (
        "<|system|>\n"
        f"{SYSTEM_PROMPT}\n"
        "<|user|>\n"
        f"{question}\n"
        "<|assistant|>\n"
    )


def extract_answer(answer_str):
    """
    GSM8K answers usually look like:
        some reasoning #### 42
    """
    match = re.search(r"####\s*(-?\d+(?:,\d{3})*)", answer_str)
    if match:
        return int(match.group(1).replace(",", ""))
    return None


def make_train_dataset(ds):
    """
    Converts raw GSM8K examples into prompt/answer examples for RL training.

    Expected raw GSM8K format:
        ds[i]["question"]
        ds[i]["answer"]
    """

    train_examples = []

    for ex in ds:
        question = ex["question"]
        answer_text = ex["answer"]

        gold_answer = extract_answer(answer_text)

        if gold_answer is None:
            continue

        train_examples.append(
            {
                "prompt": build_prompt(question),
                "question": question,
                "gold_solution": answer_text,
                "gold_answer": gold_answer,
            }
        )

    return Dataset.from_list(train_examples)


train_data = make_train_dataset("gsm8k")

TypeError: string indices must be integers, not 'str'

In [ ]:
def train_step(
    policy,
    ref_policy,
    example,
    optimizer,
    beta_kl=0.02,
    tau_projection=1.0,
    max_new_tokens=256,
):
    """
    Runs one RL training step on one example.

    example should contain:
        example["prompt"]
        example["gold_answer"]
    """

    policy.train()
    ref_policy.eval()

    # ------------------------------------------------------------
    # 1. Generate rollout from current policy
    # ------------------------------------------------------------
    with torch.no_grad():
        rollout = generate_rollout(
            policy=policy,
            prompt=example["prompt"],
            max_new_tokens=max_new_tokens,
        )

    # ------------------------------------------------------------
    # 2. Extract model answer and compute reward
    # ------------------------------------------------------------
    pred_answer = extract_answer(rollout.text)
    gold_answer = example["gold_answer"]

    reward = compute_reward(
        pred_answer=pred_answer,
        gold_answer=gold_answer,
        num_generated_tokens=len(rollout.generated_token_ids),
    )

    # ------------------------------------------------------------
    # 3. Replay rollout and compute logprob + KL
    # ------------------------------------------------------------
    total_logprob, approx_kl = compute_rollout_logprob_and_kl(
        policy=policy,
        ref_policy=ref_policy,
        rollout=rollout,
        tau_projection=tau_projection,
    )

    # ------------------------------------------------------------
    # 4. Build policy-gradient loss
    # ------------------------------------------------------------
    loss = -reward * total_logprob + beta_kl * approx_kl

    # ------------------------------------------------------------
    # 5. Backprop and update model
    # ------------------------------------------------------------
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
    optimizer.step()

    # ------------------------------------------------------------
    # 6. Return useful logging info
    # ------------------------------------------------------------
    metrics = {
        "loss": loss.detach().item(),
        "reward": float(reward),
        "pred_answer": pred_answer,
        "gold_answer": gold_answer,
        "correct": pred_answer == gold_answer,
        "num_generated_tokens": len(rollout.generated_token_ids),
        "total_logprob": total_logprob.detach().item(),
        "approx_kl": approx_kl.detach().item(),
        "rollout_text": rollout.text,
    }

    return metrics

In [ ]:
import torch


def build_optimizer(
    policy,
    lr_lora=5e-6,
    lr_gate=1e-4,
    lr_lambda=1e-3,
    weight_decay=0.1,
):
    lora_params = []
    gate_linear_params = []
    lambda_params = []
    other_trainable_params = []

    for name, p in policy.named_parameters():
        if not p.requires_grad:
            continue

        # LoRA adapter weights
        if "lora_" in name.lower():
            lora_params.append(p)

        # Lambda parameter inside HRPOGate
        elif "gate.Lambda" in name or "gate.lambda" in name.lower():
            lambda_params.append(p)

        # Other gate parameters: W_a, W_x, etc.
        elif name.startswith("gate.") or ".gate." in name:
            gate_linear_params.append(p)

        # Catch anything else trainable
        else:
            other_trainable_params.append(p)

    param_groups = []

    if len(lora_params) > 0:
        param_groups.append({
            "params": lora_params,
            "lr": lr_lora,
            "weight_decay": weight_decay,
            "name": "lora",
        })

    if len(gate_linear_params) > 0:
        param_groups.append({
            "params": gate_linear_params,
            "lr": lr_gate,
            "weight_decay": weight_decay,
            "name": "gate_linear",
        })

    if len(lambda_params) > 0:
        param_groups.append({
            "params": lambda_params,
            "lr": lr_lambda,
            "weight_decay": weight_decay,
            "name": "lambda",
        })

    if len(other_trainable_params) > 0:
        param_groups.append({
            "params": other_trainable_params,
            "lr": lr_gate,
            "weight_decay": weight_decay,
            "name": "other_trainable",
        })

    optimizer = torch.optim.AdamW(
        param_groups,
        betas=(0.9, 0.99),
        eps=1e-8,
    )

    print("Optimizer parameter groups:")
    print(f"  LoRA params:          {sum(p.numel() for p in lora_params):,}")
    print(f"  Gate linear params:   {sum(p.numel() for p in gate_linear_params):,}")
    print(f"  Lambda params:        {sum(p.numel() for p in lambda_params):,}")
    print(f"  Other trainable:      {sum(p.numel() for p in other_trainable_params):,}")

    return optimizer

In [ ]:
optimizer = build_optimizer(policy)

In [ ]:
for step, example in enumerate(train_data):
    prompt = example["prompt"]
    gold_answer = example["answer"]

    rollout = generate_rollout(
        policy=policy,
        prompt=prompt,
        max_new_tokens=256,
    )

    pred_answer = extract_answer(rollout.text)

    reward = compute_reward(
        pred_answer=pred_answer,
        gold_answer=gold_answer,
        num_generated_tokens=len(rollout.generated_token_ids),
    )

    metrics = train_step(
        policy=policy,
        ref_policy=ref_policy,
        rollout=rollout,
        optimizer=optimizer,
        reward=reward,
        beta_kl=0.02,
    )

    if step % 10 == 0:
        print(metrics)

NameError: name 'train_data' is not defined